# Domain 5: Context Management & Reliability
## Claude Certified Architect – Foundations Certification
**Weight: 15% of scored content**

This notebook covers managing conversation context, escalation patterns, error propagation,
large codebase exploration, human review workflows, and information provenance.

---

## Task 5.1: Manage Conversation Context to Preserve Critical Information

### Key Challenges

1. **Progressive Summarization Risk**: Condensing numbers, percentages, dates into vague summaries
   - ❌ "The customer wants a refund for a recent order"
   - ✅ "Customer CUST-12345 requests $1,299.00 refund for order ORD-98765 placed 2024-03-01"

2. **"Lost in the Middle" Effect**: Models reliably process info at the **beginning** and **end**
   of long inputs but may omit findings from **middle** sections

3. **Tool Result Bloat**: Tool results accumulate tokens disproportionately
   - An order lookup returns 40+ fields but only 5 are relevant

4. **Conversation History**: Complete history must be passed in subsequent API requests

### Strategies

| Strategy | Purpose |
|---|---|
| Extract "case facts" block | Persistent structured facts in each prompt |
| Trim tool outputs | Keep only relevant fields |
| Key findings at beginning | Mitigate position effects |
| Section headers | Organize detailed results |
| Structured subagent output | Metadata + key facts instead of verbose reasoning |

In [ ]:
# === CONTEXT MANAGEMENT PATTERNS ===

import json

class ContextManager:
    """
    Manages conversation context to preserve critical information.
    
    Key exam concepts:
    - Extract transactional facts into a persistent 'case facts' block
    - Trim verbose tool outputs to relevant fields
    - Place key findings at beginning of aggregated inputs
    """
    
    def __init__(self):
        self.case_facts = {}  # Persistent structured facts
        self.issues = []  # Multi-issue tracking
    
    def extract_case_facts(self, tool_result, tool_name):
        """Extract and persist transactional facts from tool results."""
        if tool_name == "get_customer":
            self.case_facts.update({
                "customer_id": tool_result.get("id"),
                "customer_name": tool_result.get("name"),
                "account_status": tool_result.get("status"),
                "verified": tool_result.get("verified", False)
            })
        elif tool_name == "lookup_order":
            self.case_facts.update({
                "order_id": tool_result.get("order_id"),
                "order_amount": tool_result.get("total"),
                "order_date": tool_result.get("date"),
                "return_eligible": tool_result.get("return_eligible")
            })
    
    def trim_tool_output(self, tool_result, relevant_fields):
        """
        Trim verbose tool outputs to only relevant fields.
        Example: Order lookup returns 40+ fields, we keep only 5.
        """
        return {k: v for k, v in tool_result.items() if k in relevant_fields}
    
    def build_context_block(self):
        """
        Build a case facts block to include in each prompt.
        This sits OUTSIDE the summarized history.
        """
        return f"""
## Case Facts (do not summarize or omit)
{json.dumps(self.case_facts, indent=2)}

## Active Issues
{json.dumps(self.issues, indent=2)}
"""

# Demo
ctx = ContextManager()

# Simulate verbose tool output (40+ fields)
verbose_order = {
    "order_id": "ORD-98765", "total": 1299.00, "date": "2024-03-01",
    "return_eligible": True, "shipping_method": "express",
    "warehouse_id": "WH-04", "pick_list_id": "PL-9876",
    "carrier_code": "UPS-1234", "box_dimensions": "12x8x6",
    "weight_lbs": 3.2, "insurance_value": 1299.00,
    "internal_notes": "Priority customer", "tax_rate": 0.08,
    # ... imagine 30 more fields
}

# Trim to relevant fields only
trimmed = ctx.trim_tool_output(verbose_order, 
    ["order_id", "total", "date", "return_eligible", "shipping_method"])

print(f"Full tool output: {len(verbose_order)} fields")
print(f"Trimmed output: {len(trimmed)} fields")
print(f"  {json.dumps(trimmed, indent=2)}")

# Extract case facts
ctx.extract_case_facts(verbose_order, "lookup_order")
print(f"\nCase facts block:")
print(ctx.build_context_block())

### Mitigating the "Lost in the Middle" Effect

```
┌─────────────────────────────────────────┐
│ KEY FINDINGS SUMMARY (beginning)    ← ✅ High attention
├─────────────────────────────────────────┤
│ == Section A: Financial Data ==         
│ Detailed results...                 ← ⚠️ Lower attention
├─────────────────────────────────────────┤
│ == Section B: Customer History ==       
│ Detailed results...                 ← ⚠️ Lower attention  
├─────────────────────────────────────────┤
│ INSTRUCTIONS FOR SYNTHESIS          ← ✅ High attention
└─────────────────────────────────────────┘
```

Place critical info at the **beginning** and **end**. Use explicit section headers in the middle.

---

## Task 5.2: Design Effective Escalation and Ambiguity Resolution Patterns

### Appropriate Escalation Triggers

| Trigger | Action |
|---|---|
| Customer explicitly requests human agent | ✅ Escalate **immediately** |
| Policy exception/gap | ✅ Escalate |
| Inability to make meaningful progress | ✅ Escalate |
| Complex case (but within capability) | ❌ Don't escalate — attempt resolution first |

### Unreliable Escalation Proxies ⚠️

- **Sentiment-based escalation**: Frustration ≠ case complexity
- **Self-reported confidence scores**: Model can't reliably calibrate its own confidence

### Ambiguity Resolution

When multiple customer matches are found:
- ✅ Ask for **additional identifiers** (email, phone, order number)
- ❌ Don't select based on heuristics (most recent, closest match)

### The Nuanced Scenario

Customer is frustrated but issue is straightforward:
1. **Acknowledge** the frustration
2. **Offer** to resolve (within capability)
3. Escalate only if customer **reiterates** preference for human

In [ ]:
# === ESCALATION DECISION ENGINE ===

from enum import Enum

class EscalationDecision(Enum):
    RESOLVE = "resolve_autonomously"
    ESCALATE = "escalate_to_human"
    CLARIFY = "ask_for_clarification"

def evaluate_escalation(context):
    """
    Escalation decision logic matching exam expectations.
    
    Key rules:
    1. Explicit human request → ALWAYS escalate immediately
    2. Policy gap → escalate
    3. Multiple matches → clarify (don't select heuristically)
    4. Complex but within capability → resolve
    5. Sentiment alone → NOT a valid escalation trigger
    """
    
    # Rule 1: Customer explicitly asks for human
    if context.get("customer_requests_human"):
        return EscalationDecision.ESCALATE, \
            "Customer explicitly requested human agent — honor immediately"
    
    # Rule 2: Policy gap or exception
    if context.get("policy_gap"):
        return EscalationDecision.ESCALATE, \
            f"Policy is silent on: {context['policy_gap']}"
    
    # Rule 3: Multiple customer matches
    if context.get("multiple_matches", 0) > 1:
        return EscalationDecision.CLARIFY, \
            "Multiple matches found — ask for additional identifiers"
    
    # Rule 4: No progress after attempts
    if context.get("attempts", 0) > 3 and not context.get("progress_made"):
        return EscalationDecision.ESCALATE, \
            "Unable to make meaningful progress after multiple attempts"
    
    # Default: Attempt resolution
    return EscalationDecision.RESOLVE, \
        "Issue is within capability — attempt autonomous resolution"

# Test scenarios
scenarios = [
    {"name": "Customer wants human", "customer_requests_human": True},
    {"name": "Competitor price match (policy gap)", "policy_gap": "competitor price matching"},
    {"name": "Multiple customer matches", "multiple_matches": 3},
    {"name": "Standard return request", "issue_type": "return"},
    {"name": "Frustrated but simple issue", "sentiment": "negative", "issue_type": "return"},
]

for s in scenarios:
    name = s.pop("name")
    decision, reason = evaluate_escalation(s)
    print(f"  {name}")
    print(f"    → {decision.value}")
    print(f"    Reason: {reason}\n")

---

## Task 5.3: Implement Error Propagation Strategies Across Multi-Agent Systems

### Structured Error Context

When a subagent fails, return:
- **Failure type** (timeout, validation, etc.)
- **Attempted query** (what was tried)
- **Partial results** (what succeeded before failure)
- **Alternative approaches** (suggestions for coordinator)

### Anti-Patterns

| Anti-Pattern | Why It's Wrong |
|---|---|
| Generic error status ("search unavailable") | Hides valuable context from coordinator |
| Silently suppressing errors (returning empty as success) | Prevents any recovery |
| Terminating entire workflow on single failure | Unnecessarily destructive |

### Access Failure vs Valid Empty Result

- **Access failure** (timeout): Needs retry/recovery decisions
- **Valid empty result** (query succeeded, no matches): Successful operation, just no data

In [ ]:
# === ERROR PROPAGATION PATTERN ===

from dataclasses import dataclass, field
from typing import Optional

@dataclass
class SubagentError:
    """Structured error context for coordinator recovery."""
    failure_type: str           # 'timeout', 'auth_failure', 'rate_limit'
    attempted_query: str        # What the subagent tried
    partial_results: list       # What succeeded before failure
    alternatives: list          # Suggested recovery approaches
    is_access_failure: bool     # True=access issue, False=valid empty result

@dataclass 
class SubagentResponse:
    """Complete subagent response with coverage annotations."""
    success: bool
    findings: list = field(default_factory=list)
    coverage: dict = field(default_factory=dict)  # What topics are well-covered vs gaps
    error: Optional[SubagentError] = None

def handle_subagent_timeout(agent_name, query, partial):
    """
    Subagent timed out. Return structured error.
    
    ✅ Return structured context → coordinator makes informed decision
    ❌ Return generic 'search unavailable' → coordinator can't decide
    ❌ Return empty results as success → hides the failure
    ❌ Terminate entire workflow → unnecessarily destructive
    """
    return SubagentResponse(
        success=False,
        findings=partial,
        coverage={
            "completed": ["visual arts", "music"],
            "failed": ["film production"],  # Gap due to timeout
            "gap_reason": "timeout during web search"
        },
        error=SubagentError(
            failure_type="timeout",
            attempted_query=query,
            partial_results=partial,
            alternatives=[
                "Retry with narrower query scope",
                "Use cached results from previous session",
                "Proceed with partial results and annotate gaps"
            ],
            is_access_failure=True
        )
    )

# Demo: Subagent timeout scenario
response = handle_subagent_timeout(
    agent_name="web_search",
    query="AI impact on film production industry",
    partial=[{"topic": "AI in visual arts", "sources": 5}]
)

print("Subagent Error Response:")
print(f"  Success: {response.success}")
print(f"  Partial results: {len(response.findings)} findings")
print(f"  Coverage gaps: {response.coverage.get('failed', [])}")
print(f"  Error type: {response.error.failure_type}")
print(f"  Alternatives: {response.error.alternatives}")
print(f"  Is access failure: {response.error.is_access_failure}")

---

## Task 5.4: Manage Context Effectively in Large Codebase Exploration

### Context Degradation Signs

In extended sessions, models start:
- Giving **inconsistent answers**
- Referencing **"typical patterns"** rather than specific classes discovered earlier
- Losing track of discovered information

### Mitigation Strategies

| Strategy | How |
|---|---|
| **Scratchpad files** | Persist key findings across context boundaries |
| **Subagent delegation** | Isolate verbose exploration, keep main agent lean |
| **Structured state persistence** | Export agent state for crash recovery |
| **`/compact`** | Reduce context usage during extended sessions |

### Crash Recovery Pattern

1. Each agent exports state to a known location (manifest)
2. Coordinator loads manifest on resume
3. Injects state into agent prompts

In [ ]:
# === LARGE CODEBASE EXPLORATION PATTERNS ===

import json

class ScratchpadManager:
    """
    Persist key findings across context boundaries.
    Counteracts context degradation in extended sessions.
    """
    
    def __init__(self, filepath="/tmp/exploration_scratchpad.json"):
        self.filepath = filepath
        self.findings = {}
    
    def record_finding(self, category, key, value):
        """Record a key finding for later reference."""
        if category not in self.findings:
            self.findings[category] = {}
        self.findings[category][key] = value
    
    def save(self):
        """Persist findings to file."""
        with open(self.filepath, 'w') as f:
            json.dump(self.findings, f, indent=2)
    
    def load(self):
        """Load findings from previous session."""
        try:
            with open(self.filepath) as f:
                self.findings = json.load(f)
        except FileNotFoundError:
            self.findings = {}
    
    def get_summary(self):
        """Generate a summary for injection into agent prompts."""
        return json.dumps(self.findings, indent=2)


class CrashRecoveryManifest:
    """
    Structured state persistence for crash recovery.
    Each agent exports state; coordinator loads on resume.
    """
    
    def __init__(self):
        self.agent_states = {}
    
    def export_agent_state(self, agent_name, state):
        self.agent_states[agent_name] = {
            "completed_tasks": state.get("completed", []),
            "pending_tasks": state.get("pending", []),
            "key_findings": state.get("findings", []),
            "last_checkpoint": state.get("checkpoint", "start")
        }
    
    def build_resume_prompt(self, agent_name):
        """Generate a prompt for resuming an agent after crash."""
        state = self.agent_states.get(agent_name, {})
        return f"""
## Resuming from crash — Agent: {agent_name}

### Previous State
Completed: {state.get('completed_tasks', [])}
Pending: {state.get('pending_tasks', [])}
Last checkpoint: {state.get('last_checkpoint', 'unknown')}

### Key Findings (from before crash)
{json.dumps(state.get('key_findings', []), indent=2)}

Continue from the last checkpoint. Do not re-do completed tasks.
"""

# Demo
scratchpad = ScratchpadManager()
scratchpad.record_finding("architecture", "entry_point", "src/main.py")
scratchpad.record_finding("architecture", "db_layer", "src/db/repository.py")
scratchpad.record_finding("dependencies", "auth_module", "Uses JWT with bcrypt")
scratchpad.record_finding("tests", "coverage", "42% — gaps in API layer")

print("Scratchpad findings:")
print(scratchpad.get_summary())

# Crash recovery
manifest = CrashRecoveryManifest()
manifest.export_agent_state("analysis_agent", {
    "completed": ["map_structure", "identify_entry_points"],
    "pending": ["trace_data_flow", "analyze_error_handling"],
    "findings": [{"type": "architecture", "detail": "Monolithic with 3 service boundaries"}],
    "checkpoint": "structure_mapped"
})
print("\n" + manifest.build_resume_prompt("analysis_agent"))

---

## Task 5.5: Design Human Review Workflows and Confidence Calibration

### The Hidden Risk in Aggregate Metrics

> 97% overall accuracy may **mask poor performance** on specific document types or fields.

### Strategies

| Strategy | Purpose |
|---|---|
| **Stratified random sampling** | Measure error rates in high-confidence extractions |
| **Field-level confidence scores** | Route review attention to low-confidence fields |
| **Accuracy by document type** | Verify consistent performance across segments |
| **Calibration with labeled sets** | Ensure confidence scores are meaningful |

### Review Routing Logic

```
High confidence + simple document → Auto-approve
High confidence + rare document type → Sample for audit
Low confidence → Human review
Ambiguous/contradictory source → Human review (priority)
```

In [ ]:
# === HUMAN REVIEW ROUTING ===

from dataclasses import dataclass

@dataclass
class ExtractionResult:
    document_id: str
    document_type: str
    fields: dict
    field_confidence: dict  # field_name → confidence score
    has_contradictions: bool

class ReviewRouter:
    """
    Route extractions based on confidence, document type, and contradictions.
    
    Key exam points:
    - Don't trust aggregate accuracy alone
    - Stratified sampling catches novel error patterns
    - Verify accuracy by document type AND field before automating
    """
    
    def __init__(self, confidence_threshold=0.85):
        self.threshold = confidence_threshold
        self.review_stats = {"auto": 0, "human": 0, "sampled": 0}
    
    def route(self, result: ExtractionResult):
        # Priority 1: Contradictions always go to human review
        if result.has_contradictions:
            self.review_stats["human"] += 1
            return "HUMAN_REVIEW", "Contradictory source data detected"
        
        # Priority 2: Any low-confidence field triggers human review
        low_conf_fields = [
            f for f, c in result.field_confidence.items() 
            if c < self.threshold
        ]
        if low_conf_fields:
            self.review_stats["human"] += 1
            return "HUMAN_REVIEW", f"Low confidence on: {low_conf_fields}"
        
        # Priority 3: Stratified sampling of high-confidence results
        # Sample rare document types more frequently
        import random
        sample_rate = 0.05  # 5% baseline
        if result.document_type in ["handwritten", "scanned", "multilingual"]:
            sample_rate = 0.20  # 20% for rare types
        
        if random.random() < sample_rate:
            self.review_stats["sampled"] += 1
            return "SAMPLED_REVIEW", f"Stratified sample ({result.document_type})"
        
        # Default: Auto-approve
        self.review_stats["auto"] += 1
        return "AUTO_APPROVED", "High confidence across all fields"

# Demo
router = ReviewRouter(confidence_threshold=0.85)

test_results = [
    ExtractionResult("DOC-001", "standard_invoice", {}, 
                     {"amount": 0.95, "date": 0.98, "vendor": 0.92}, False),
    ExtractionResult("DOC-002", "handwritten_receipt", {},
                     {"amount": 0.72, "date": 0.65, "vendor": 0.80}, False),
    ExtractionResult("DOC-003", "standard_invoice", {},
                     {"amount": 0.96, "date": 0.94, "vendor": 0.91}, True),  # Contradictions!
]

for result in test_results:
    decision, reason = router.route(result)
    print(f"  {result.document_id} ({result.document_type}):")
    print(f"    → {decision}: {reason}\n")

---

## Task 5.6: Preserve Information Provenance and Handle Uncertainty in Multi-Source Synthesis

### The Provenance Problem

During summarization, **source attribution is lost** when findings are compressed
without preserving claim-source mappings.

### Key Principles

1. **Claim-source mappings**: Every claim must reference its source
2. **Conflicting statistics**: Annotate conflicts with source attribution (don't arbitrarily pick one)
3. **Temporal data**: Include publication/collection dates to prevent temporal confusion
4. **Content-appropriate rendering**: Financial data → tables, news → prose, technical → structured lists

### Required Subagent Output Structure

```json
{
  "claim": "AI adoption in film VFX grew 73%",
  "source_url": "https://example.com/report",
  "document_name": "Film Industry AI Report 2024",
  "publication_date": "2024-06-15",
  "relevant_excerpt": "According to our survey of 200 studios...",
  "confidence": "well-established"
}
```

In [ ]:
# === INFORMATION PROVENANCE TRACKING ===

from dataclasses import dataclass, field
from typing import Optional

@dataclass
class ClaimSourceMapping:
    """Structured claim with full provenance."""
    claim: str
    source_url: str
    document_name: str
    publication_date: Optional[str]
    relevant_excerpt: str
    confidence: str  # 'well-established', 'contested', 'single-source'

@dataclass
class SynthesisReport:
    """Report with provenance preservation and conflict handling."""
    topic: str
    well_established: list = field(default_factory=list)
    contested: list = field(default_factory=list)
    coverage_gaps: list = field(default_factory=list)
    
    def add_finding(self, claims: list):
        """Add claims, handling conflicts."""
        # Group by claim topic
        claim_groups = {}
        for claim in claims:
            key = claim.claim[:50]  # Rough grouping
            claim_groups.setdefault(key, []).append(claim)
        
        for key, group in claim_groups.items():
            if len(group) == 1:
                self.well_established.append(group[0])
            else:
                # Multiple sources — check for conflict
                self.contested.append({
                    "topic": key,
                    "claims": group,
                    "note": "Multiple sources with different values — both preserved"
                })
    
    def render(self):
        """Render report with appropriate section structure."""
        output = f"# Synthesis Report: {self.topic}\n\n"
        
        output += "## Well-Established Findings\n"
        for finding in self.well_established:
            output += f"- {finding.claim}\n"
            output += f"  Source: {finding.document_name} ({finding.publication_date})\n\n"
        
        if self.contested:
            output += "## Contested/Conflicting Findings\n"
            for conflict in self.contested:
                output += f"- Topic: {conflict['topic']}\n"
                for claim in conflict['claims']:
                    output += f"  - {claim.claim} (Source: {claim.document_name})\n"
                output += f"  Note: {conflict['note']}\n\n"
        
        if self.coverage_gaps:
            output += "## Coverage Gaps\n"
            for gap in self.coverage_gaps:
                output += f"- {gap}\n"
        
        return output

# Demo: Conflicting statistics from credible sources
report = SynthesisReport(topic="AI in Creative Industries")

claims = [
    ClaimSourceMapping(
        claim="AI adoption in film VFX reached 73% of studios",
        source_url="https://example.com/film-report",
        document_name="Film Industry AI Report 2024",
        publication_date="2024-06-15",
        relevant_excerpt="Survey of 200 VFX studios worldwide",
        confidence="well-established"
    ),
    ClaimSourceMapping(
        claim="AI adoption in film VFX reached 58% of studios",
        source_url="https://example.com/tech-survey",
        document_name="Tech Adoption Survey 2024",
        publication_date="2024-03-01",
        relevant_excerpt="Survey of 500 studios (broader definition)",
        confidence="well-established"
    ),
    ClaimSourceMapping(
        claim="AI music composition tools grew 340% in 2024",
        source_url="https://example.com/music-ai",
        document_name="Music Tech Trends",
        publication_date="2024-12-01",
        relevant_excerpt="Analysis of DAW plugin downloads",
        confidence="well-established"
    )
]

report.add_finding(claims)
report.coverage_gaps = ["AI in writing/publishing (no sources found)"]

print(report.render())

---

## Domain 5 Summary & Exam Preparation

### Key Concepts to Remember

1. **Case facts block**: Extract transactional facts into a persistent block (outside summary)
2. **"Lost in the middle"**: Key info at beginning and end; section headers in middle
3. **Trim tool outputs**: Keep only relevant fields before they accumulate
4. **Escalation triggers**: Customer request (immediate), policy gaps, inability to progress
5. **NOT valid triggers**: Sentiment analysis, self-reported confidence scores
6. **Multiple matches**: Ask for identifiers, don't select heuristically
7. **Error propagation**: Structured context (type + query + partial results + alternatives)
8. **Anti-patterns**: Generic errors, silent suppression, workflow termination
9. **Scratchpad files**: Persist findings across context boundaries
10. **`/compact`**: Reduce context during extended exploration
11. **Aggregate metrics can deceive**: Check accuracy by document type AND field
12. **Provenance**: Claim-source mappings preserved through synthesis
13. **Conflicts**: Annotate with both sources, don't arbitrarily pick one

### Sample Exam Question

> Web search subagent times out. What error propagation approach enables intelligent recovery?
>
> ✅ Structured error context (failure type, attempted query, partial results, alternatives)
>
> ❌ Automatic retry with generic "search unavailable" status
>
> ❌ Return empty result set marked as successful
>
> ❌ Terminate entire research workflow